# Gaussian Processes

Companion notebook for the [Gaussian Processes lesson](https://ml-viz-ruby.vercel.app/courses/bayesian-methods/02-gaussian-processes).

We implement **GP regression from scratch** with an RBF kernel: sample functions from the prior,
condition on a few observations to get the closed-form posterior mean and variance, and watch the
length-scale reshape the fit. Pure NumPy + Matplotlib.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({
    'figure.facecolor': '#0f1117', 'axes.facecolor': '#1a1d27',
    'axes.edgecolor': '#444', 'axes.labelcolor': '#ccc',
    'xtick.color': '#888', 'ytick.color': '#888',
    'text.color': '#eee', 'grid.color': '#333', 'lines.linewidth': 2,
})
rng = np.random.default_rng(1)

## 1 — The RBF kernel and prior samples

The kernel sets how correlated two outputs are. Sampling from N(0, K) over a grid gives random
*functions* — all smooth, with wiggliness set by the length-scale.

In [ ]:
def rbf_kernel(A, B, ell=1.0, sig=1.0):
    d2 = (A[:, None] - B[None, :]) ** 2
    return sig**2 * np.exp(-d2 / (2 * ell**2))

xs = np.linspace(-5, 5, 120)
K = rbf_kernel(xs, xs, ell=1.0) + 1e-9 * np.eye(len(xs))
L = np.linalg.cholesky(K)
fig, ax = plt.subplots(figsize=(8, 3.5))
for _ in range(5):
    ax.plot(xs, L @ rng.normal(size=len(xs)), alpha=0.8)
ax.set_title('5 functions sampled from the GP prior (RBF, ℓ=1)')
ax.set_xlabel('x'); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 2 — The closed-form GP posterior

Condition on observations: μ* = K*ᵀ(K+σ²I)⁻¹y,  Σ* = K** − K*ᵀ(K+σ²I)⁻¹K*. We plot the mean and the
±2σ band — note it pinches at the data and widens away from it.

In [ ]:
def gp_posterior(X, y, Xs, ell=1.0, sig=1.0, noise=0.04):
    K = rbf_kernel(X, X, ell, sig) + noise * np.eye(len(X))
    Ks = rbf_kernel(X, Xs, ell, sig)
    Kss = rbf_kernel(Xs, Xs, ell, sig)
    Kinv = np.linalg.inv(K)
    mu = Ks.T @ Kinv @ y
    cov = Kss - Ks.T @ Kinv @ Ks
    return mu, np.sqrt(np.clip(np.diag(cov), 1e-9, None))

Xo = np.array([-3.0, -1.8, -0.5, 1.2, 2.6])
yo = np.array([-1.2, 0.9, 0.4, -0.8, 1.1])
mu, sd = gp_posterior(Xo, yo, xs, ell=1.0)

fig, ax = plt.subplots(figsize=(8, 4))
ax.fill_between(xs, mu - 2*sd, mu + 2*sd, color='#6366f1', alpha=0.3, label='±2σ posterior')
ax.plot(xs, mu, color='#818cf8', label='posterior mean')
ax.scatter(Xo, yo, color='#2dd4bf', zorder=5, label='observations')
ax.set_title('GP posterior: confident at data, uncertain away from it')
ax.legend(facecolor='#1a1d27', edgecolor='#444'); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()
print('σ at an observation (x=-0.5):', round(gp_posterior(Xo, yo, np.array([-0.5]))[1][0], 3))
print('σ far from data  (x=5.0):   ', round(gp_posterior(Xo, yo, np.array([5.0]))[1][0], 3))

## 3 — The length-scale reshapes the fit

Small ℓ → wiggly, uncertain between points; large ℓ → smooth, confident. There's no single 'right'
value — in practice it's chosen by maximizing the marginal likelihood.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.2), sharey=True)
for ax, ell in zip(axes, [0.4, 1.0, 2.5]):
    mu, sd = gp_posterior(Xo, yo, xs, ell=ell)
    ax.fill_between(xs, mu-2*sd, mu+2*sd, color='#6366f1', alpha=0.3)
    ax.plot(xs, mu, color='#818cf8')
    ax.scatter(Xo, yo, color='#2dd4bf', zorder=5)
    ax.set_title(f'ℓ = {ell}'); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## ✏️ Your turn

**Exercise.** Implement `rbf(a, b, ell, sig)` for two scalars and `posterior_var(X, Xs, ell, sig,
noise)` returning the GP posterior variance at each test point in `Xs`:
diag(K** − K*ᵀ(K+σ²I)⁻¹K*). (The mean needs y; the variance does not — it depends only on *where*
the data is, not its values.)

In [ ]:
def rbf(a, b, ell=1.0, sig=1.0):
    # TODO(you): scalar RBF kernel sig^2 * exp(-(a-b)^2 / (2 ell^2))
    return ...

def posterior_var(X, Xs, ell=1.0, sig=1.0, noise=0.04):
    # TODO(you): return the vector of posterior variances at the points in Xs
    return ...

In [ ]:
# This assert cell passes silently when your implementation is correct.
assert np.isclose(rbf(1.0, 1.0), 1.0)                      # zero distance -> max covariance
assert rbf(0.0, 3.0) < rbf(0.0, 1.0)                       # farther -> less correlated
v = posterior_var(Xo, xs)
# variance is small near an observation, large far away
near = posterior_var(Xo, np.array([-0.5]))[0]
far  = posterior_var(Xo, np.array([5.0]))[0]
assert far > near
assert np.all(v >= -1e-6)                                    # variances are non-negative
print(f'\u2713 kernel and posterior variance correct (near={near:.3f} < far={far:.3f})')

<details>
<summary>Solution</summary>

```python
def rbf(a, b, ell=1.0, sig=1.0):
    return sig**2 * np.exp(-(a - b)**2 / (2 * ell**2))

def posterior_var(X, Xs, ell=1.0, sig=1.0, noise=0.04):
    K = rbf_kernel(X, X, ell, sig) + noise * np.eye(len(X))
    Ks = rbf_kernel(X, Xs, ell, sig)
    Kss = rbf_kernel(Xs, Xs, ell, sig)
    return np.diag(Kss - Ks.T @ np.linalg.inv(K) @ Ks)
```

The variance depends only on the *locations* of the data, not the observed y-values — which is why
active learning can pick where to sample next (highest variance) before seeing any label there.

</details>